In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from backend.app.services.data_services import (
    get_available_indicators,
    get_available_strategies,
    get_strategies_metadata,
    get_available_tickers,
    _read_csv,
    fetch_data_to_df,
    get_indicators_metadata,
    _load_and_resample_data
)
import importlib
import backend.app.core.Strategies as st

import backend.app.services.cache_service as cs
import backend.app.services.data_services as ds
import backend.app.services.plotting_services as ps
import backend.app.services.backtest_services as bs

# importlib.reload(cs)
# importlib.reload(ds)
# importlib.reload(ps)
# importlib.reload(bs)

import asyncio
import pandas as pd
import numpy as np
import vectorbt as vbt

In [3]:
ticker_name = "TATA CONSULTANCY SERVICES"
df = await ds._read_csv(ticker_name=ticker_name)
print(df.head(2))
print(df.tail(2))
await cs.set_data(df=df, ticker= ticker_name)

                        open     high      low    close  volume
date_time                                                      
2021-11-01 09:15:00  3437.95  3437.95  3421.10  3429.80   81979
2021-11-01 09:16:00  3429.30  3434.95  3427.15  3434.95   28995
                       open    high     low   close  volume
date_time                                                  
2025-10-31 15:28:00  3060.4  3063.0  3057.1  3057.1   18200
2025-10-31 15:29:00  3057.1  3060.0  3051.8  3060.0    5292


{'message': 'Data loaded successfully with key: data:TATA CONSULTANCY SERVICES'}

In [4]:
cs.get_key_list()
# await cs.delete_data("SOLUSD")
# ds.get_available_tickers()
# ds.get_available_strategies()

['data:TATA CONSULTANCY SERVICES']

In [5]:
resolution = "15m"
start_date = "01/11/2021 09:15:00"
end_date = "30/10/2024 09:15:00"

df = await ds._load_and_resample_data(ticker_name, resolution, start_date, end_date)
print(df.shape)
price = df['close']
df.head()

(18502, 5)


d:\OneDrive - iitgn.ac.in\Desktop\HedgeOne-Quant\backend\app\services\data_services.py:54: FutureWarning:

'T' is deprecated and will be removed in a future version, please use 'min' instead.



,open,high,low,close,volume
2021-11-01 09:15:00,3437.95,3438.85,3412.95,3426.45,271905
2021-11-01 09:30:00,3425.25,3433.80,3424.50,3430.60,73662
2021-11-01 09:45:00,3430.90,3446.80,3430.10,3440.00,120704
2021-11-01 10:00:00,3440.25,3444.40,3430.50,3433.75,61136
2021-11-01 10:15:00,3433.75,3439.00,3428.00,3439.00,67726


In [65]:
ema_fast = vbt.MA.run(price, 9)
ema_slow = vbt.MA.run(price, 26)

entries  = ema_fast.ma_crossed_above(ema_slow)
exits    = ema_fast.ma_crossed_below(ema_slow)

p = vbt.Portfolio.from_signals(
    close=price,
    entries=entries,
    exits=exits,
    init_cash=100000,
    fees=0.0,
    slippage=0.0,
    freq=resolution
)

In [66]:
print(round(p.total_return()*100, 2), round(p.stats()["Win Rate [%]"], 2), round(p.sharpe_ratio(),2))

31.4 37.31 1.82


In [49]:
print(round(p.total_return()*100, 2), round(p.stats()["Win Rate [%]"], 2), round(p.sharpe_ratio(),2))

31.4 37.31 1.82


In [73]:
k_values = [round(x * 0.1, 2) for x in range(0,22,2)] 
k_values

[0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

In [ ]:
import vectorbt as vbt
import pandas as pd
import csv
import os

# Check if CSV exists
file_exists = os.path.isfile("logs1.csv")

# Open CSV once in append mode
with open("logs1.csv", mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    
    # Write header only if file is new
    if not file_exists:
        writer.writerow(["Fast", "Slow", "SL", "TP", "TSL", "ATR", "WinRate", "Return", "Sharpe"])
    
    # Loop through parameter grid
    for i in range(1,25,3):
        for j in range(i+1,45,3):
            k_values = [round(x * 0.1, 2) for x in range(0,22,3)] 
            l_values = [round(x * 0.1, 2) for x in range(0,22,3)]
            m_values = [round(x * 0.1, 2) for x in range(0,22,3)]
            
            for k in k_values:
                for l in l_values:
                    for m in m_values:
                        for n in range(3, 20, 4):

                            ema_fast = vbt.MA.run(price, i)
                            ema_slow = vbt.MA.run(price, j)

                            entries = ema_fast.ma_crossed_above(ema_slow)
                            exits = ema_fast.ma_crossed_below(ema_slow)

                            atr = vbt.ATR.run(df["high"], df["low"], df["close"], n).atr
                            sl_dist = (k * atr) / price
                            tp_dist = (l * atr) / price
                            tsl_dist = (m * atr) / price

                            p = vbt.Portfolio.from_signals(
                                close=price,
                                entries=entries,
                                exits=exits,
                                sl_stop=sl_dist,
                                tp_stop=tp_dist,
                                sl_trail=tsl_dist,
                                init_cash=100000,
                                fees=0.0,
                                freq=resolution,
                                accumulate=False
                            )

                            win_rate = round(p.stats()["Win Rate [%]"], 2)
                            sharp = round(p.sharpe_ratio(), 2)
                            returns = round(p.total_return() * 100, 2)

                            if (win_rate>45 and sharp>2 and returns>45) or (win_rate>60) or (sharp>3.5) or (returns>60):
                                print(f"Fast: {i} Slow: {j} SL: {k} TP: {l} TSL: {m} ATR: {n} WR: {win_rate} Return: {returns} Sharp: {sharp}")
                            
                            writer.writerow([i, j, k, l, m, n, win_rate, returns, sharp])

Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 0.0 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 0.3 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 0.6 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 0.9 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 1.2 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 1.5 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 1.8 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.2 TP: 0.0 TSL: 2.1 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.5 TP: 0.0 TSL: 0.0 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.5 TP: 0.0 TSL: 0.3 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.5 TP: 0.0 TSL: 0.6 ATR: 3 WR: 60.06 Return: -2.13 Sharp: -0.18
Fast: 1 Slow: 20 SL: 1.5 TP: 0.0 TSL: 0.9 ATR: 3 WR: 6

In [32]:
pairs = [(7,8),(11,12),(12,13),(22,23),(102,105)]
for i,j in pairs:
    ema_fast = vbt.MA.run(price, i)
    ema_slow = vbt.MA.run(price, j)

    # trend_ok = price > ema_htf.ma
    entries  = ema_fast.ma_crossed_above(ema_slow)
    exits    = ema_fast.ma_crossed_below(ema_slow)

    p = vbt.Portfolio.from_signals(
        close=price,
        entries=entries,
        exits=exits,
        init_cash=100000,
        fees=0.0,
        slippage=0.0,
        freq=resolution
    )
    win_rate = round(p.stats()["Win Rate [%]"], 2)
    sharp = round(p.sharpe_ratio(),2)
    returns = round(p.total_return()*100, 2)

    print(f"Fast: {i} Slow: {j} WR: {win_rate} Return: {returns} Sharp: {sharp}")

Fast: 7 Slow: 8 WR: 40.85 Return: -1.27 Sharp: -0.04
Fast: 11 Slow: 12 WR: 44.09 Return: -10.54 Sharp: -1.88
Fast: 12 Slow: 13 WR: 45.67 Return: 3.26 Sharp: 0.74
Fast: 22 Slow: 23 WR: 44.68 Return: -1.17 Sharp: -0.05
Fast: 102 Slow: 105 WR: 52.24 Return: -9.26 Sharp: -1.76


In [39]:
print(round(p.total_return()*100, 2), round(p.stats()["Win Rate [%]"], 2), round(p.sharpe_ratio(),2))

31.4 37.31 1.82


In [17]:
p.plot()

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': '#1f77b4'},
              'name': 'Close',
              'showlegend': True,
              'type': 'scatter',
              'uid': '65e8b92a-64e3-47c8-8551-9584736d63f6',
              'x': array(['2021-11-01T09:15:00.000000000', '2021-11-01T09:30:00.000000000',
                          '2021-11-01T09:45:00.000000000', ..., '2024-10-29T15:00:00.000000000',
                          '2024-10-29T15:15:00.000000000', '2024-10-30T09:15:00.000000000'],
                         dtype='datetime64[ns]'),
              'xaxis': 'x',
              'y': {'bdata': ('ZmZmZubEqkAzMzMzM82qQAAAAAAA4K' ... 'mZGdOvQAAAAAAA4K9AAAAAAMAAsEA='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAAAAA40RHzpJM8QAAAAAAAAA' ... 'AAAACgiED8bc4HWAlAQAAAAAAAAAAA'),
                             'dtype': 'f8',
                             'shape': '395, 3'},
              'ho